# Vi-PPE V5 Gemma 3 / Llama 3.1 Judge Runs

Notebook nay chay `google/gemma-3-4b-it` va `meta-llama/Llama-3.1-8B-Instruct` tren V5. Mac dinh ca hai dung bf16, batch size 16 tren A100 40GB de so sanh voi moc Qwen2.5 7B Instruct bf16 batch 16 da chay on.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import yaml

ROOT = Path('/content/Vi-PPE-mini')
if ROOT.exists():
    %cd /content/Vi-PPE-mini
else:
    print('Repo path /content/Vi-PPE-mini not found; staying in current directory')


In [ ]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes accelerate sentencepiece protobuf


In [ ]:
import torch
print('cuda_available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
if torch.cuda.is_available():
    print('bf16_supported:', torch.cuda.is_bf16_supported())
    print('total_gpu_gb:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


## 2. HF Token, Dataset, And Config Check

Gemma va Llama tren Hugging Face co license gate. Hay accept license tren Hugging Face va set `HF_TOKEN` trong Colab secrets truoc khi chay.

In [ ]:
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token and not os.environ.get('HF_TOKEN'):
        os.environ['HF_TOKEN'] = hf_token
except Exception:
    pass

print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')))


In [ ]:
TEST_DATASET = 'data/processed/pairs_test_llm_v4.jsonl'
BIAS_DATASET = 'data/processed/bias_subset_llm_v4.jsonl'

for path in [TEST_DATASET, BIAS_DATASET]:
    p = Path(path)
    print(path, 'exists=', p.exists(), 'rows=', sum(1 for _ in p.open(encoding='utf-8')) if p.exists() else None)

with open('configs/models.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

MODEL_RUNS = [
    ('gemma3_4b_it_a100_40gb_b16', 'gemma3_4b_it'),
    ('llama31_8b_instruct_a100_40gb_b16', 'llama31_8b_instruct'),
]

for model_key, slug in MODEL_RUNS:
    print('\n===', model_key, '===')
    print(cfg['models'][model_key])


## 3. Model Compatibility Smoke Check

In [ ]:
from transformers import AutoConfig, AutoTokenizer

for model_key, slug in MODEL_RUNS:
    model_cfg = cfg['models'][model_key]
    print('\n===', model_key, '===')
    model_config = AutoConfig.from_pretrained(model_cfg['model_id'], trust_remote_code=bool(model_cfg.get('trust_remote_code', False)))
    print('model_type:', getattr(model_config, 'model_type', None))
    print('loader:', model_cfg.get('loader', 'causal_lm'))
    if model_cfg.get('loader') == 'gemma3_conditional_generation':
        from transformers import AutoProcessor
        processor = AutoProcessor.from_pretrained(model_cfg['model_id'], trust_remote_code=bool(model_cfg.get('trust_remote_code', False)))
        print('processor:', type(processor).__name__)
        print('has_tokenizer:', hasattr(processor, 'tokenizer'))
    else:
        tok = AutoTokenizer.from_pretrained(model_cfg['model_id'], trust_remote_code=bool(model_cfg.get('trust_remote_code', False)))
        print('tokenizer:', type(tok).__name__)
        print('has_chat_template:', bool(getattr(tok, 'chat_template', None)))


## 4. Helpers

In [ ]:
def run_cmd(args):
    print('$', ' '.join(args))
    subprocess.run(args, check=True)


def run_judge(model_key, template, run_id, dataset, limit=None):
    args = [
        sys.executable, 'scripts/05_run_judge.py',
        '--dataset', dataset,
        '--model', model_key,
        '--template', template,
        '--run-id', run_id,
        '--resume',
    ]
    if limit is not None:
        args += ['--limit', str(limit)]
    run_cmd(args)


def compute_metrics(run_id, dataset, bias=False, limit=None):
    args = [
        sys.executable, 'scripts/06_compute_metrics.py',
        '--judgments', f'results/judgments/{run_id}.jsonl',
        '--dataset', dataset,
        '--run-id', run_id,
    ]
    if bias:
        args.append('--bias')
    if limit is not None:
        args += ['--limit', str(limit)]
    run_cmd(args)


## 5. Smoke Runs

In [ ]:
SMOKE_LIMIT = 8

for model_key, slug in MODEL_RUNS:
    baseline_id = f'{slug}_baseline_test_llm_v5_a100_b16_smoke'
    criteria_id = f'{slug}_criteria_test_llm_v5_a100_b16_smoke'

    run_judge(model_key, 'baseline_generic_vi', baseline_id, TEST_DATASET, limit=SMOKE_LIMIT)
    compute_metrics(baseline_id, TEST_DATASET, limit=SMOKE_LIMIT)

    run_judge(model_key, 'auto_criteria_by_domain', criteria_id, TEST_DATASET, limit=SMOKE_LIMIT)
    compute_metrics(criteria_id, TEST_DATASET, limit=SMOKE_LIMIT)


## 6. Full Core Runs

In [ ]:
for model_key, slug in MODEL_RUNS:
    baseline_id = f'{slug}_baseline_test_llm_v5_a100_b16'
    criteria_id = f'{slug}_criteria_test_llm_v5_a100_b16'

    run_judge(model_key, 'baseline_generic_vi', baseline_id, TEST_DATASET)
    run_judge(model_key, 'auto_criteria_by_domain', criteria_id, TEST_DATASET)

    compute_metrics(baseline_id, TEST_DATASET)
    compute_metrics(criteria_id, TEST_DATASET)


## 7. Bias Runs

In [ ]:
for model_key, slug in MODEL_RUNS:
    bias_baseline_id = f'{slug}_bias_baseline_llm_v5_a100_b16'
    bias_mitigated_id = f'{slug}_bias_mitigated_llm_v5_a100_b16'

    run_judge(model_key, 'baseline_generic_vi', bias_baseline_id, BIAS_DATASET)
    run_judge(model_key, 'criteria_bias_mitigated_vi', bias_mitigated_id, BIAS_DATASET)

    compute_metrics(bias_baseline_id, BIAS_DATASET, bias=True)
    compute_metrics(bias_mitigated_id, BIAS_DATASET, bias=True)


## 8. Summary

In [ ]:
run_ids = []
for _, slug in MODEL_RUNS:
    run_ids.extend([
        f'{slug}_baseline_test_llm_v5_a100_b16',
        f'{slug}_criteria_test_llm_v5_a100_b16',
        f'{slug}_bias_baseline_llm_v5_a100_b16',
        f'{slug}_bias_mitigated_llm_v5_a100_b16',
    ])

for run_id in run_ids:
    path = Path('results/metrics') / f'{run_id}.json'
    print('\n===', run_id, '===')
    if not path.exists():
        print('missing metrics:', path)
        continue
    metrics = json.loads(path.read_text(encoding='utf-8'))
    for key in ['num_pairs', 'num_judgments', 'parse_success_rate', 'pairwise_accuracy', 'macro_accuracy', 'lower_bound_domain_score', 'swap_consistency', 'invalid_count', 'inconsistent_count', 'domain_accuracy']:
        if key in metrics:
            print(f'{key}: {metrics[key]}')


## 9. Parse Failure Debug

In [ ]:
for run_id in run_ids:
    path = Path('results/judgments') / f'{run_id}.jsonl'
    if not path.exists():
        continue
    bad = []
    for line in path.read_text(encoding='utf-8').splitlines():
        row = json.loads(line)
        if not row.get('parsed'):
            bad.append(row)
    print('\n===', run_id, 'bad:', len(bad), '===')
    for row in bad[:3]:
        print('pair_id:', row.get('pair_id'), 'order:', row.get('order'))
        print((row.get('raw_output') or '')[:1200])


## 10. Backup To Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/Vi-PPE-mini/results_gemma_llama_v5_a100_b16 /content/drive/MyDrive/Vi-PPE-mini/checkpoints_gemma_llama_v5_a100_b16
!cp -r results/judgments /content/drive/MyDrive/Vi-PPE-mini/results_gemma_llama_v5_a100_b16/
!cp -r results/metrics /content/drive/MyDrive/Vi-PPE-mini/results_gemma_llama_v5_a100_b16/
!cp -r results/figures /content/drive/MyDrive/Vi-PPE-mini/results_gemma_llama_v5_a100_b16/ || true
!tar -czf /content/drive/MyDrive/Vi-PPE-mini/checkpoints_gemma_llama_v5_a100_b16/after_gemma_llama_v5_$(date +%Y%m%d_%H%M%S).tar.gz results configs data/processed reports prompts src scripts notebooks/Vi_PPE_V5_Gemma_Llama.ipynb
